# Multimodal RAG with Captioning

Caption images with a vision LLM, then index captions next to the text so one text index serves both.

**Extra requirements:** matplotlib, `ollama pull llava`

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import ollama

figures = make_figures()          # renders 3 sample chart PNGs into data/figures
print([f.name for f in figures])

## Caption every image
A vision model turns each chart into searchable text.

In [ ]:
def caption(path):
    reply = ollama.chat(model=VISION_MODEL, messages=[{
        "role": "user",
        "content": "Describe this chart in detail: title, axes, every value and the main takeaway.",
        "images": [str(path)],
    }])
    return reply["message"]["content"].strip()

captions = {f.name: caption(f) for f in figures}
for name, cap in captions.items():
    print(name, "->", cap[:200], "\n")

## One index for text and images
Caption chunks keep a pointer to the source image so you can show it with the answer.

In [ ]:
mm_store = VectorStore().add(chunks, [{"type": "text"} for _ in chunks])
mm_store.add(list(captions.values()), [{"type": "image", "file": n} for n in captions])

query = "Which retrieval method achieves the best recall?"
hits = mm_store.search(query, 3)
for h in hits: print(h["meta"], f"{h['score']:.3f}")
print(answer(query, [h["text"] for h in hits]))

In [ ]:
from IPython.display import Image, display
for h in hits:
    if h["meta"]["type"] == "image":
        display(Image(filename=str(DATA / "figures" / h["meta"]["file"])))
        break